In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict

class AODE:
    def __init__(self, m  = 30):
        self.m = m  # 阈值常数，用于判断超父
        self.priors = {}  # 存储P(c, xi)的概率
        self.conditional_probs = {}  # 存储P(xj|c, xi)的条件概率
        self.gaussian_params = {}  # 存储连续特征的高斯分布参数
        self.super_parent_features = {}  # 存储每个特征的超父取值
        self.classes = None  # 存储类别标签
        self.discrete_features = []  # 离散特征索引
        self.continuous_features = []  # 连续特征索引
        self.n_features = None  # 特征数量

    def _is_continuous(self, values):
        """判断特征是否为连续型"""
        return any(isinstance(val, float) for val in values)

    def _calculate_class_count(self, data):
        """统计类别数目"""
        class_count = {}
        for val in data:
            if val not in class_count:
                class_count[val] = 0
            class_count[val] += 1
        return class_count

    def _get_super_parent_values(self, values):
        """获取满足超父条件的取值"""
        super_parent_feature = []
        class_count = self._calculate_class_count(values)
        for i in class_count.keys():
            if class_count[i] >= self.m:
                super_parent_feature.append(i)
        return super_parent_feature

    def _calculate_gaussian_params(self, values):
        """计算连续特征的高斯分布参数"""
        mean_val = np.mean(values)
        var_val = np.var(values, ddof=0)  # 总体方差
        return mean_val, var_val

    def fit(self, X, y):
        """训练AODE模型"""
        X = np.array(X)
        y = np.array(y)
        n_samples, self.n_features = X.shape
        
        # 获取类别信息
        self.classes = np.unique(y)
        class_count = self._calculate_class_count(y)
        
        # 识别离散和连续特征
        for j in range(self.n_features):
            if self._is_continuous(X[:, j]):
                self.continuous_features.append(j)
            else:
                self.discrete_features.append(j)
                # 存储每个离散特征的超父取值
                self.super_parent_features[j] = self._get_super_parent_values(X[:, j])
        
        # 计算P(c,xi)和P(xj|c,xi)
        for c in self.classes:
            c_mask = (y == c)
            n_c = np.sum(c_mask)
            
            # 处理离散特征作为超父的情况
            for i in self.discrete_features:
                super_values = self.super_parent_features[i]
                
                for value in super_values:
                    # 获取满足y=c且xi=value的样本
                    mask = c_mask & (X[:, i] == value)
                    X_c_xi = X[mask]
                    n_c_xi = len(X_c_xi)
                    
                    # 计算P(c,xi) - 公式(7.24)
                    n_values_i = len(self.super_parent_features[i])
                    prior = (n_c_xi + 1) / (n_samples + len(self.classes) * n_values_i)
                    self.priors[(c, i, value)] = prior
                    
                    # 计算P(xj|c,xi) - 特征的条件概率
                    for j in range(self.n_features):
                        if j == i:  # 跳过超父特征自身
                            continue
                            
                        if j in self.discrete_features:  # 离散特征
                            # 获取特征j的所有可能取值
                            values_j = np.unique(X[:, j])
                            probs = {}
                            
                            for v_j in values_j:
                                # 计算条件概率,公式(7.25)
                                D_c_xi_xj = np.sum(X_c_xi[:, j] == v_j)
                                n_j = len(values_j)
                                prob = (D_c_xi_xj + 1) / (n_c_xi + n_j)
                                probs[v_j] = prob
                            self.conditional_probs[(c, i, value, j)] = probs
                        
                        else:  # 连续特征
                            # 确保有足够的样本计算高斯参数
                            if n_c_xi > 1:
                                mean_val, var_val = self._calculate_gaussian_params(X_c_xi[:, j])
                                self.gaussian_params[(c, i, value, j)] = (mean_val, var_val)
                            # 样本不足时使用类别c的全局分布
                            elif n_c > 1: 
                                mean_val, var_val = self._calculate_gaussian_params(X[c_mask, j])
                                self.gaussian_params[(c, i, value, j)] = (mean_val, var_val)
                            else:
                                # 使用全体数据的均值和方差
                                mean_val, var_val = self._calculate_gaussian_params(X[:, j])
                                self.gaussian_params[(c, i, value, j)] = (mean_val, var_val)
        return self

    def _gaussian_prob(self, x, mean, var):
        """计算高斯分布的概率密度"""
        if var <= 0:  # 方差为0时避免除以0
            return 1.0 if x == mean else 0.0
        return np.exp(-(x - mean)**2 / (2 * var)) / np.sqrt(2 * np.pi * var)

    def predict(self, X):
        """预测样本类别"""
        X = np.array(X)
        predictions = []
        
        for x in X:
            class_scores = {}
            
            for c in self.classes:
                score = 0.0
                super_parent_contributions = 0
                
                # 考虑每个可能的超父特征
                for i in self.discrete_features:
                    if i >= len(x):  # 检查索引范围
                        continue
                        
                    # 检查当前值是否是超父取值
                    if x[i] in self.super_parent_features[i]:
                        # 获取P(c, xi)
                        prior_key = (c, i, x[i])
                        prior = self.priors.get(prior_key, 0.0)
                        print("P(c=%s, x_%d=%s) = %.4f" % (c, i, x[i], prior))
                        # 如果先验为0，则跳过
                        if prior <= 0:
                            continue
                            
                        # 计算各特征的条件概率乘积
                        cond_prob_product = prior
                        
                        for j in range(self.n_features):
                            if j == i:  # 跳过超父特征自身
                                continue
                                
                            if j < len(x):  # 检查索引范围
                                if j in self.discrete_features:  # 离散特征
                                    cond_probs = self.conditional_probs.get((c, i, x[i], j), {})
                                    prob = cond_probs.get(x[j], 1e-5)  # 避免0概率
                                else:  # 连续特征
                                    params = self.gaussian_params.get((c, i, x[i], j), (0, 1))
                                    prob = self._gaussian_prob(x[j], *params)
                                print("P(x_%d=%s|c=%s, x_%d=%s) = %.4f" % (j, x[j], c, i, x[i], prob))
                                cond_prob_product *= prob
                        
                        # 累加超父贡献
                        super_parent_contributions += cond_prob_product
                # 如果至少有一个超父有贡献，则累加到类得分
                if super_parent_contributions > 0:
                    class_scores[c] = super_parent_contributions
                print('P(c=%s|x)∝%f'%(c, super_parent_contributions))
                print('-------------------------------------------')
            # 选择得分最高的类别
            if class_scores:
                predictions.append(max(class_scores, key=class_scores.get))
            else:
                # 没有任何超父贡献时，选择频率最高的类别作为后备
                predictions.append(max(self.classes, key=lambda c: np.sum(np.array(y) == c)))
        
        return np.array(predictions)
    
if __name__ == '__main__':
    # 加载西瓜数据集3.0
    data = pd.read_csv('../Data/watermelon3.0.csv', encoding='ansi')
    
    # 提取特征和标签
    X = data.iloc[:, 1:-1].values
    y = data.iloc[:, -1].values
    
    # 训练AODE模型
    aode = AODE(m=0)  # 设置超父阈值
    aode.fit(X, y)
    
    # 预测第一行样本
    sample = X[0:1]
    print("预测结果:", aode.predict(sample))
    print("实际标签:", y[0])

P(c=否, x_0=青绿) = 0.1739
P(x_1=蜷缩|c=否, x_0=青绿) = 0.3333
P(x_2=浊响|c=否, x_0=青绿) = 0.3333
P(x_3=清晰|c=否, x_0=青绿) = 0.3333
P(x_4=凹陷|c=否, x_0=青绿) = 0.3333
P(x_5=硬滑|c=否, x_0=青绿) = 0.6000
P(x_6=0.697|c=否, x_0=青绿) = 1.4088
P(x_7=0.46|c=否, x_0=青绿) = 0.0010
P(c=否, x_1=蜷缩) = 0.1739
P(x_0=青绿|c=否, x_1=蜷缩) = 0.3333
P(x_2=浊响|c=否, x_1=蜷缩) = 0.5000
P(x_3=清晰|c=否, x_1=蜷缩) = 0.1667
P(x_4=凹陷|c=否, x_1=蜷缩) = 0.1667
P(x_5=硬滑|c=否, x_1=蜷缩) = 0.6000
P(x_6=0.697|c=否, x_1=蜷缩) = 1.6566
P(x_7=0.46|c=否, x_1=蜷缩) = 0.0000
P(c=否, x_2=浊响) = 0.2174
P(x_0=青绿|c=否, x_2=浊响) = 0.2857
P(x_1=蜷缩|c=否, x_2=浊响) = 0.4286
P(x_3=清晰|c=否, x_2=浊响) = 0.2857
P(x_4=凹陷|c=否, x_2=浊响) = 0.2857
P(x_5=硬滑|c=否, x_2=浊响) = 0.5000
P(x_6=0.697|c=否, x_2=浊响) = 0.8332
P(x_7=0.46|c=否, x_2=浊响) = 0.2010
P(c=否, x_3=清晰) = 0.1304
P(x_0=青绿|c=否, x_3=清晰) = 0.4000
P(x_1=蜷缩|c=否, x_3=清晰) = 0.2000
P(x_2=浊响|c=否, x_3=清晰) = 0.4000
P(x_4=凹陷|c=否, x_3=清晰) = 0.2000
P(x_5=硬滑|c=否, x_3=清晰) = 0.2500
P(x_6=0.697|c=否, x_3=清晰) = 0.0000
P(x_7=0.46|c=否, x_3=清晰) = 0.1778
P(c=否, x_4=凹陷) =